[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ContextLab/llm-course/blob/main/slides/week7/video_audio_diffusion_demo.ipynb)

# Diffusion models for video and audio

**PSYC 51.17: Models of language and communication**
**Week 7 — Lecture 23 companion**

---

## Overview

In Lecture 22 we saw how diffusion models generate images. Lecture 23 extends
this to **video** (spacetime patches) and **audio** (spectrogram-based diffusion).

In this notebook we will:
1. Generate short videos from text using a text-to-video diffusion model
2. Generate audio from text using AudioLDM 2 (the CLAP-conditioned system from lecture)
3. Visualize audio as spectrograms — seeing why "audio diffusion is image diffusion"
4. Explore generation parameters and discuss ethical implications

**Requirements:** A GPU runtime is needed for generation. In Colab, go to
**Runtime → Change runtime type → T4 GPU** (free tier works).

In [ ]:
# Install required packages (for Colab)
!pip install -q diffusers transformers accelerate torch scipy matplotlib numpy librosa

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import torch
import IPython.display as ipd

# Reproducibility
np.random.seed(42)
torch.manual_seed(42)

print(f"PyTorch version: {torch.__version__}")
if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    gpu_mem = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"GPU: {gpu_name} ({gpu_mem:.1f} GB)")
else:
    print("No GPU detected. Go to Runtime > Change runtime type > T4 GPU")

---

## Part 1: Text-to-video generation

In lecture we discussed how **Sora** extends diffusion to video by treating
frames as 3D spacetime patches. Sora itself is not publicly available, but
smaller open-source text-to-video models use similar principles.

We will use **ModelScope Text-to-Video** (`damo-vilab/text-to-video-ms-1.7b`),
a 1.7B parameter model that:
- Operates in a latent space (like Stable Diffusion for images)
- Adds a **temporal dimension** to the UNet so it can denoise across frames
- Generates 16 frames (~2 seconds at 8 fps) within Colab's memory limits

The architecture mirrors what we discussed: compress → denoise → decode,
but extended from 2D (height × width) to 3D (height × width × time).

In [ ]:
if torch.cuda.is_available():
    from diffusers import DiffusionPipeline
    from diffusers.utils import export_to_video

    # Load the text-to-video pipeline
    # For higher quality (but different API), try: Wan-AI/Wan2.1-T2V-1.3B-Diffusers
    pipe_video = DiffusionPipeline.from_pretrained(
        "damo-vilab/text-to-video-ms-1.7b",
        torch_dtype=torch.float16,
        variant="fp16",
    )

    # Memory optimizations for Colab's T4 (15 GB VRAM)
    pipe_video.enable_model_cpu_offload()  # move components to CPU when not in use
    pipe_video.enable_vae_slicing()         # decode one frame at a time

    print("\u2713 Text-to-video pipeline loaded!")
else:
    print("No GPU — skipping video model loading.")
    print("Enable a GPU runtime to run this section.")

In [ ]:
if torch.cuda.is_available():
    prompt = "A cat walking across a sunlit kitchen floor"

    print(f"Generating video for: \"{prompt}\"")
    print("This takes ~1-2 minutes on a T4 GPU...")

    video_frames = pipe_video(
        prompt,
        num_inference_steps=25,
        num_frames=16,
    ).frames[0]

    # Save and display the video
    video_path = export_to_video(video_frames, "generated_video.mp4", fps=8)
    print(f"\u2713 Video saved to {video_path}")

    # Display inline
    ipd.display(ipd.Video("generated_video.mp4", embed=True, width=512))
else:
    print("No GPU available. Here's what would happen:")
    print("  - The model generates 16 frames (~2 seconds at 8 fps)")
    print("  - Each frame is 256x256 pixels")
    print("  - Generation takes ~1-2 minutes on a T4 GPU")

In [ ]:
if torch.cuda.is_available():
    # Show individual frames to see temporal evolution
    n_show = min(8, len(video_frames))
    indices = np.linspace(0, len(video_frames) - 1, n_show, dtype=int)

    fig, axes = plt.subplots(1, n_show, figsize=(16, 3))
    for i, idx in enumerate(indices):
        axes[i].imshow(video_frames[idx])
        axes[i].set_title(f"Frame {idx}", fontsize=10)
        axes[i].axis("off")

    fig.suptitle(f"Video frames: \"{prompt}\"", fontsize=13)
    plt.tight_layout()
    plt.show()

    # Free GPU memory before loading the audio model
    del pipe_video
    torch.cuda.empty_cache()
    print("\u2713 Video pipeline unloaded to free GPU memory")

---

## Part 2: Text-to-audio generation

As we discussed in lecture, audio diffusion works by applying latent diffusion
to **spectrograms** — 2D representations of sound (frequency × time):

1. A **VAE** encodes the spectrogram into a latent space
2. A diffusion model denoises in latent space, conditioned on text via **CLAP**
   (Contrastive Language-Audio Pretraining — the audio equivalent of CLIP)
3. The VAE decodes back to a spectrogram
4. A **vocoder** (HiFi-GAN) converts the spectrogram to an audio waveform

We will use **AudioLDM 2** — the exact system from the lecture slides. It can
generate speech, music, and sound effects, all from text descriptions.

In [ ]:
if torch.cuda.is_available():
    from diffusers import AudioLDM2Pipeline
    import scipy

    pipe_audio = AudioLDM2Pipeline.from_pretrained(
        "cvssp/audioldm2",
        torch_dtype=torch.float16,
    )
    pipe_audio.enable_model_cpu_offload()

    print("\u2713 AudioLDM 2 pipeline loaded!")
else:
    print("No GPU — skipping audio model loading.")

In [ ]:
if torch.cuda.is_available():
    audio_prompt = "A thunderstorm with heavy rain and distant thunder"

    print(f"Generating audio for: \"{audio_prompt}\"")
    print("This takes ~30-60 seconds...")

    result = pipe_audio(
        audio_prompt,
        num_inference_steps=50,
        audio_length_in_s=10.0,
        return_dict=False,
    )
    # return_dict=False returns a tuple; first element is the audio array
    audio = result[0][0]

    # AudioLDM 2 outputs at 16 kHz
    sample_rate = 16000

    # Save as WAV
    scipy.io.wavfile.write("generated_audio.wav", rate=sample_rate, data=audio)
    print(f"\u2713 Audio saved ({len(audio) / sample_rate:.1f} seconds)")

    # Play inline
    ipd.display(ipd.Audio(audio, rate=sample_rate))
else:
    print("No GPU available. AudioLDM 2 would generate:")
    print("  - 10 seconds of audio at 16 kHz")
    print("  - Conditioned on text via CLAP embeddings")
    print("  - Generation takes ~30-60 seconds on a T4")

### Visualizing the spectrogram

The key insight from lecture: by converting audio into a spectrogram (a 2D image),
we can reuse the entire image diffusion toolkit. Let's see what our generated
audio looks like as a mel spectrogram — this is the representation the diffusion
model actually works with internally.

In [ ]:
if torch.cuda.is_available():
    import librosa
    import librosa.display

    # Compute mel spectrogram
    S = librosa.feature.melspectrogram(
        y=audio, sr=sample_rate, n_mels=128, fmax=8000
    )
    S_dB = librosa.power_to_db(S, ref=np.max)

    fig, axes = plt.subplots(1, 2, figsize=(14, 4))

    # Waveform
    time = np.arange(len(audio)) / sample_rate
    axes[0].plot(time, audio, color="#003C6C", linewidth=0.5)
    axes[0].set_xlabel("Time (s)")
    axes[0].set_ylabel("Amplitude")
    axes[0].set_title("Waveform")

    # Mel spectrogram
    img = librosa.display.specshow(
        S_dB, sr=sample_rate, x_axis="time", y_axis="mel",
        fmax=8000, ax=axes[1], cmap="magma"
    )
    axes[1].set_title("Mel spectrogram")
    fig.colorbar(img, ax=axes[1], format="%+2.0f dB")

    fig.suptitle(f"Generated audio: \"{audio_prompt}\"", fontsize=13)
    plt.tight_layout()
    plt.show()

    print("The mel spectrogram on the right is essentially a 2D image —")
    print("that's why image diffusion techniques transfer directly to audio!")
else:
    print("No GPU available. The spectrogram visualization shows:")
    print("  Left: the raw audio waveform (amplitude over time)")
    print("  Right: the mel spectrogram (frequency bands over time)")
    print("  The spectrogram is a 2D image — enabling image diffusion for audio.")

---

## Part 3: Exploring generation parameters

Let's experiment with how different parameters affect audio generation.
Try changing the prompts and settings below to build intuition for what
these models can and cannot do.

In [ ]:
if torch.cuda.is_available():
    # Try different prompts — sound effects, music, and ambient sounds
    prompts = [
        "A dog barking in a park with birds chirping",
        "Soft piano jazz music in a cozy cafe",
        "Ocean waves crashing on a rocky shore",
    ]

    fig, axes = plt.subplots(len(prompts), 2, figsize=(14, 3 * len(prompts)))

    for i, p in enumerate(prompts):
        print(f"Generating: \"{p}\"...")
        result = pipe_audio(
            p,
            num_inference_steps=30,      # fewer steps = faster but lower quality
            audio_length_in_s=5.0,       # shorter clips for speed
            negative_prompt="low quality, noise, distortion",
            return_dict=False,
        )
        a = result[0][0]

        # Waveform
        t = np.arange(len(a)) / sample_rate
        axes[i, 0].plot(t, a, color="#003C6C", linewidth=0.5)
        axes[i, 0].set_ylabel("Amplitude")
        axes[i, 0].set_title(p, fontsize=10)

        # Spectrogram
        S = librosa.feature.melspectrogram(y=a, sr=sample_rate, n_mels=128, fmax=8000)
        S_dB = librosa.power_to_db(S, ref=np.max)
        librosa.display.specshow(
            S_dB, sr=sample_rate, x_axis="time", y_axis="mel",
            fmax=8000, ax=axes[i, 1], cmap="magma"
        )

        # Play each clip
        print(f"  Playing: {p}")
        ipd.display(ipd.Audio(a, rate=sample_rate))

    axes[-1, 0].set_xlabel("Time (s)")
    axes[-1, 1].set_xlabel("Time (s)")
    fig.suptitle("Comparing audio types: effects, music, ambient", fontsize=13)
    plt.tight_layout()
    plt.show()
else:
    print("No GPU available.")
    print("With a GPU, this cell generates three different audio types:")
    print("  1. Sound effects (dog barking, birds)")
    print("  2. Music (piano jazz)")
    print("  3. Ambient sounds (ocean waves)")
    print("  Each is shown as a waveform and spectrogram side by side.")

---

## Part 4: Discussion questions

1. **Quality gap**: Compare the video and audio we generated to production
   systems like Sora or commercial music generators. What are the main
   differences? How much of the gap is model size vs. training data vs.
   architecture?

2. **The spectrogram trick**: We saw that audio diffusion is essentially
   image diffusion on spectrograms. What other modalities could be
   "converted to images" and generated this way? What about EEG signals,
   weather data, or DNA sequences?

3. **Temporal coherence**: Watch the video frames carefully. How well does
   the model maintain consistency across time? How does this compare to
   what Sora achieves with spacetime patches?

4. **Ethical implications**: Audio generation can produce realistic speech
   and music. Consider:
   - How could generated audio be used for voice cloning or impersonation?
   - Should AI-generated music be eligible for copyright?
   - What safeguards should text-to-audio systems have?

5. **Multimodal generation**: We've now seen text → image, text → video,
   and text → audio, all using variants of diffusion. What would it take
   to build a single model that generates all three modalities together
   (e.g., a video *with* synchronized sound)?

---

### References

- [Liu et al. (2023)](https://arxiv.org/abs/2308.05734) "AudioLDM 2: Learning Holistic Audio Generation with Self-supervised Pretraining" — the AudioLDM 2 system used in this notebook
- [Wang et al. (2023)](https://arxiv.org/abs/2308.06571) "ModelScope Text-to-Video Technical Report" — the video generation model used here
- [OpenAI (2024)](https://openai.com/research/video-generation-models-as-world-simulators) "Video Generation Models as World Simulators" — Sora
- [Evans et al. (2024)](https://arxiv.org/abs/2404.10301) "Stable Audio Open" — alternative audio generation approach

*Notebook for PSYC 51.17, Dartmouth College.*